# Fig. — CORDIS-ADMM convergence

Two-panel convergence figure for the journal paper's Simulation Results section.

- **(a)** primal/dual consensus residuals vs ADMM iteration (log-y), overlaying the
  **fixed-ρ** run (the working default) and the **adaptive-ρ** run (the ablation that
  swings / doesn't settle — this justifies `adaptive_rho=false` being the shipped default).
- **(b)** for the default run: the SOC slack `max_u ε_u` collapsing toward zero (the ALM
  driving feasibility) and the worst-user **min-SINR** crossing the γ requirement.

Generate the two traces first with `bash run_traces.sh` (see README), then run all cells.
All plotting logic lives in `build_fig_convergence.py`; this notebook is the interactive
front-end so the figure and the headless build stay identical.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────
# Make the sibling builder module importable, and make `cordis` importable
# by letting the builder auto-detect the repo root.
import sys
from pathlib import Path
FIG_DIR = Path.cwd()
if str(FIG_DIR) not in sys.path:
    sys.path.insert(0, str(FIG_DIR))

import numpy as np
import matplotlib.pyplot as plt

import build_fig_convergence as B   # shared logic (load_trace, build_figure, ...)

# Set USE_TEX=False if pdflatex isn't on PATH (e.g. a headless compute node).
USE_TEX = True

## 1. Locate the two traces

By default we read `data/trace_fixed` and `data/trace_adaptive` (written by
`run_traces.sh`). Override the paths here to re-render an older pair of runs.

In [ ]:
FIXED_DIR    = FIG_DIR / 'data' / 'trace_fixed'
ADAPTIVE_DIR = FIG_DIR / 'data' / 'trace_adaptive'   # set to None to plot the default only

fixed, md_fixed = B.load_trace(FIXED_DIR)
adaptive = None
if ADAPTIVE_DIR is not None and (Path(ADAPTIVE_DIR) / 'manifest.json').exists():
    adaptive, _ = B.load_trace(ADAPTIVE_DIR)

gamma_db = B.extract_gamma_db(md_fixed, fallback=5.0)
print(f'gamma            = {gamma_db:g} dB')
print(f'fixed best_iter  = {getattr(fixed, "best_iter", None)}')
print(f'fixed n_iters    = {len(fixed.primal_res_history)}')
print(f'adaptive present = {adaptive is not None}')

## 2. Build the figure

In [ ]:
fig = B.build_figure(fixed, adaptive, gamma_db=gamma_db, use_tex=USE_TEX)
plt.show()

### 2-1. Override the figure style

In [ ]:

use_tex=USE_TEX
"""Build the two-panel convergence figure.

`fixed` / `adaptive` are LoadedADMMResult objects (adaptive may be
None — then panel (a) shows the default run only). Returns the
matplotlib Figure."""
import matplotlib
if not use_tex:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from cordis.plotting import apply_paper_style, figsize

apply_paper_style(use_latex=use_tex)
if not use_tex:
    matplotlib.rcParams["text.usetex"] = False


SET_TITLE = False

C_FIXED = "#1f3b73"   # navy   — fixed-rho (default)
C_ADAPT = "#d1791f"   # orange — adaptive-rho (ablation)
C_SLACK = "#6a51a3"   # purple — SOC slack
C_SINR = "#1f3b73"    # navy   — min-SINR
C_GUIDE = "#666666"   # grey   — best-iter / gamma guides

DATA_LW = 1.5
GUIDE_LW = 1.0

# Define the IEEE standard font size
IEEE_FONT_SIZE = 8
plt.rcParams.update({
    # Base font size (affects default text and titles)
    "font.size": IEEE_FONT_SIZE,
    # Axis labels (e.g., 'ADMM iteration', 'min-SINR [dB]')
    "axes.labelsize": IEEE_FONT_SIZE+2,
    # Axis tick labels (the numbers on the axes)
    "xtick.labelsize": IEEE_FONT_SIZE,
    "ytick.labelsize": IEEE_FONT_SIZE,
    # Legend font size
    "legend.fontsize": IEEE_FONT_SIZE,
    # Legend title font size (if you ever use legend titles)
    "legend.title_fontsize": IEEE_FONT_SIZE,
    # Figure titles (if you ever use fig.suptitle)
    "figure.titlesize": IEEE_FONT_SIZE,
    # Subplot titles (e.g., if you switch SET_TITLE to True)
    "axes.titlesize": IEEE_FONT_SIZE,
})



fig, (axA, axB) = plt.subplots(
    1, 2, figsize=figsize("double", aspect=2.4),
)

# ── Panel (a): residual convergence ──────────────────────────────
def _plot_res(admm, color, tag):
    pr = np.asarray(admm.primal_res_history, dtype=float)
    du = np.asarray(admm.dual_res_history, dtype=float)
    it = np.arange(1, len(pr) + 1)
    axA.semilogy(it, np.maximum(pr, 1e-30), color=color, ls="-",
                 lw=DATA_LW, label=rf"{tag}: $r_{{\mathrm{{pri}}}}$",
                 marker=None, markersize=3, markevery=10)
    axA.semilogy(np.arange(1, len(du) + 1), np.maximum(du, 1e-30),
                 color=color, ls="--", lw=DATA_LW,
                 label=rf"{tag}: $r_{{\mathrm{{dual}}}}$",
                 marker=None, markersize=3, markevery=10)

_plot_res(fixed, C_FIXED, r"fixed $\rho$")
if adaptive is not None:
    _plot_res(adaptive, C_ADAPT, r"adaptive $\rho$")

bi = getattr(fixed, "best_iter", None)
if bi:
    axA.axvline(int(bi), color=C_GUIDE, ls=":", lw=GUIDE_LW,
                label=f"best iter ({int(bi)})")
axA.set_xlabel("ADMM iteration")
axA.set_ylabel("Consensus Residual")
if SET_TITLE: axA.set_title("(a) Residual convergence")
axA.grid(True, which="both", alpha=0.3)
axA.legend(loc="upper right", ncol=1)

# ── Panel (b): slack -> 0 and min-SINR crossing gamma ────────────
slack = B._max_slack(fixed)
sinr_db = B._min_sinr_db(fixed)

handles, labels = [], []
if slack is not None:
    it_s = np.arange(1, len(slack) + 1)
    (h_sl,) = axB.semilogy(it_s, np.maximum(slack, 1e-30),
                           color=C_SLACK, ls="-", lw=DATA_LW,
                           label=r"max$_u\,\varepsilon_u$ (SOC slack)")
    handles.append(h_sl); labels.append(h_sl.get_label())
axB.set_xlabel("ADMM iteration")
axB.set_ylabel(r"max$_u\,\varepsilon_u$  (SOC slack)", color=C_SLACK)
axB.tick_params(axis="y", labelcolor=C_SLACK)
axB.grid(True, which="both", alpha=0.3)

axB2 = axB.twinx()
if sinr_db is not None:
    it_g = np.arange(1, len(sinr_db) + 1)
    (h_g,) = axB2.plot(it_g, sinr_db, color=C_SINR, ls="-", lw=DATA_LW,
                       label=r"min-SINR",
                       marker=None, markersize=3, markevery=10)
    handles.append(h_g); labels.append(h_g.get_label())
h_gamma = axB2.axhline(gamma_db, color="k", ls="--", lw=GUIDE_LW,
                       label=rf"$\gamma = {gamma_db:g}$ dB")
handles.append(h_gamma); labels.append(h_gamma.get_label())
axB2.set_ylabel("min-SINR [dB]", color=C_SINR)
axB2.tick_params(axis="y", labelcolor=C_SINR)

if bi:
    axB.axvline(int(bi), color=C_GUIDE, ls=":", lw=GUIDE_LW)
if SET_TITLE: axB.set_title("(b) Feasibility dynamics (default run)")
axB2.legend(handles, labels, loc="best")

fig.tight_layout()

## 3. Quick numeric sanity check

Confirms the slack actually collapses and the min-SINR ends above γ on the default run.

In [ ]:
ms  = B._min_sinr_db(fixed)
sl  = B._max_slack(fixed)
if ms is not None:
    print(f'min-SINR: first={ms[0]:+.2f} dB  final={ms[-1]:+.2f} dB  (gamma={gamma_db:g})')
if sl is not None:
    print(f'max slack: first={sl[0]:.2e}  final={sl[-1]:.2e}')
bi = getattr(fixed, 'best_iter', None)
if bi and ms is not None:
    print(f'min-SINR at best_iter {bi}: {ms[min(int(bi)-1, len(ms)-1)]:+.2f} dB')

## 4. Save to `paper/figures/`

Writes the committed PDF (and a PNG preview). Re-run the headless build any time with:
`python3 build_fig_convergence.py` (add `--no-tex` on a node without pdflatex).

In [ ]:
from cordis.plotting import save_figure

tag = '_final'
out_stem = B.FIGURES_OUT / str('fig_convergence' + tag)
out_stem.parent.mkdir(parents=True, exist_ok=True)
paths = save_figure(fig, out_stem, formats=('pdf',))
png = out_stem.with_suffix('.png'); fig.savefig(png, dpi=200, bbox_inches='tight')
for p in list(paths) + [png]:
    print('wrote', p)